# Train Mountain General 9.6 km Residual U-Net

Train from the packaged four-domain dataset and evaluate whether the residual U-Net improves mass-solver winds on terrain held out from training. The default run trains on Berthoud, Breck/Tenmile, and Keystone, then evaluates Loveland/A-Basin.


In [ ]:
from pathlib import Path
import csv
import json
import os
import shutil
import subprocess
import sys
import zipfile
from collections import Counter

IN_COLAB = 'google.colab' in sys.modules or Path('/content').exists()
DRIVE_ROOT = Path('/content/drive/MyDrive/windninja_ml') if IN_COLAB else Path.cwd() / 'ml/residual_unet/outputs/colab_local'
LOCAL_ROOT = Path('/content/windninja_ml') if IN_COLAB else Path.cwd()
LOCAL_DATA_ROOT = Path('/content/data') if IN_COLAB else Path.cwd() / 'ml/residual_unet/data/processed'

GCP_PROJECT = os.environ.get('GCP_PROJECT', 'spring-nova-475120-r0')
GCS_BUCKET = os.environ.get('GCS_BUCKET', 'mwn-ml-general-9p6-spring-nova-475120-r0')
USE_GCS_ARTIFACTS = True
SYNC_RESULTS_TO_GCS = True
FORCE_DOWNLOAD_CODE = True
FORCE_DOWNLOAD_DATA = False
FORCE_UNPACK_CODE = True
FORCE_UNPACK_DATA = False

DATASET_NAME = 'mountain_general_9p6_monthly_controlled_v1'
RUN_NAME = 'mountain_general_9p6_holdout_loveland_v1'
CONFIG_NAME = f'{RUN_NAME}.yaml'

TRAIN_BATCH_SIZE = 32
EVAL_BATCH_SIZE = 32
NUM_WORKERS = 2
PREFETCH_FACTOR = 4
PROGRESS_EVERY = 100

UPLOAD_DIR = DRIVE_ROOT / 'drive_upload'
DATASET_ZIP = UPLOAD_DIR / f'{DATASET_NAME}_dataset.zip'
CODE_ZIP = UPLOAD_DIR / 'residual_unet_code.zip'

REPO_DIR = LOCAL_ROOT / 'repo'
DATA_DIR = LOCAL_DATA_ROOT / DATASET_NAME
RESULT_DIR = DRIVE_ROOT / 'results' / RUN_NAME
CHECKPOINT_DIR = RESULT_DIR / 'checkpoints'
LOG_CSV = RESULT_DIR / 'train_log.csv'
EVAL_ROOT = RESULT_DIR / 'eval'

print('IN_COLAB', IN_COLAB)
print('RUN_NAME', RUN_NAME)
print('DATASET_ZIP', DATASET_ZIP)
print('CODE_ZIP', CODE_ZIP)
print('RESULT_DIR', RESULT_DIR)


## Authenticate, Download, And Unpack

This cell mounts Drive, optionally pulls the packaged code and dataset from GCS, and unpacks them onto local Colab disk. Code is force-unpacked so notebook reruns do not keep stale training scripts.


In [ ]:
if IN_COLAB:
    from google.colab import auth, drive
    drive.mount('/content/drive')
    if USE_GCS_ARTIFACTS:
        auth.authenticate_user()
        subprocess.run(['gcloud', 'config', 'set', 'project', GCP_PROJECT], check=True)

UPLOAD_DIR.mkdir(parents=True, exist_ok=True)
REPO_DIR.parent.mkdir(parents=True, exist_ok=True)
LOCAL_DATA_ROOT.mkdir(parents=True, exist_ok=True)


def copy_from_gcs_if_missing(blob_name: str, local_path: Path, *, force: bool = False) -> None:
    if local_path.exists() and not force:
        print(f'Found {local_path}')
        return
    if not USE_GCS_ARTIFACTS:
        raise FileNotFoundError(local_path)
    gcs_path = f'gs://{GCS_BUCKET}/drive_upload/{blob_name}'
    print(f'Copying {gcs_path} -> {local_path}')
    subprocess.run(['gcloud', 'storage', 'cp', gcs_path, str(local_path)], check=True)


def unpack_zip(zip_path: Path, dest_dir: Path, marker: Path, *, force: bool = False) -> None:
    if force and dest_dir.exists():
        print(f'Removing stale unpacked directory: {dest_dir}')
        shutil.rmtree(dest_dir)
    if marker.exists():
        print(f'Already unpacked: {marker}')
        return
    assert zip_path.exists(), zip_path
    print(f'Unpacking {zip_path} -> {dest_dir}')
    with zipfile.ZipFile(zip_path) as archive:
        archive.extractall(dest_dir)

copy_from_gcs_if_missing('residual_unet_code.zip', CODE_ZIP, force=FORCE_DOWNLOAD_CODE)
copy_from_gcs_if_missing(f'{DATASET_NAME}_dataset.zip', DATASET_ZIP, force=FORCE_DOWNLOAD_DATA)

unpack_zip(CODE_ZIP, REPO_DIR, REPO_DIR / 'ml/residual_unet/train.py', force=FORCE_UNPACK_CODE)
unpack_zip(DATASET_ZIP, LOCAL_DATA_ROOT, DATA_DIR / 'manifest.csv', force=FORCE_UNPACK_DATA)

print('Repo:', REPO_DIR)
print('Data:', DATA_DIR)


## Install And Check Runtime


In [ ]:
requirements = REPO_DIR / 'ml/residual_unet/requirements.txt'
if IN_COLAB:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(requirements)], check=True)
else:
    print('Skipping install outside Colab')

import torch
print('torch', torch.__version__)
print('cuda_available', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu', torch.cuda.get_device_name(0))
    print('gpu_memory_gb', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))


## Inspect Dataset And Split


In [ ]:
sys.path.insert(0, str(REPO_DIR))
from ml.residual_unet.config import load_config
from ml.residual_unet.dataset import filter_rows

summary = json.loads((DATA_DIR / 'dataset_summary.json').read_text())
rows = list(csv.DictReader((DATA_DIR / 'manifest.csv').open()))
config_path = REPO_DIR / 'ml/residual_unet/configs' / CONFIG_NAME
config = load_config(config_path)
data_cfg = config['data']

train_excludes = [item.strip() for item in str(data_cfg.get('train_exclude_source_datasets', '')).split(',') if item.strip()]
val_excludes = [item.strip() for item in str(data_cfg.get('val_exclude_source_datasets', '')).split(',') if item.strip()]
train_rows = filter_rows(rows, 'train', exclude_source_datasets=train_excludes)
val_rows = filter_rows(rows, 'val', exclude_source_datasets=val_excludes)

print(json.dumps(summary, indent=2)[:4000])
print('total manifest rows:', len(rows))
print('training rows for this run:', len(train_rows))
print('validation rows for this run:', len(val_rows))
print('held out from train/val:', train_excludes or 'none')
print('by source:')
for source, count in Counter(row['source_dataset'] for row in rows).most_common():
    print(f'  {source}: {count}')


## Train Holdout Model

Default: hold out Loveland/A-Basin. To run a different held-out terrain, change `RUN_NAME` and `CONFIG_NAME` in the setup cell to one of:

- `mountain_general_9p6_holdout_loveland_v1`
- `mountain_general_9p6_holdout_keystone_v1`
- `mountain_general_9p6_holdout_breck_v1`
- `mountain_general_9p6_monthly_controlled_v1` for the normal mixed-source split


In [ ]:
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
LOG_CSV.parent.mkdir(parents=True, exist_ok=True)
config_path = REPO_DIR / 'ml/residual_unet/configs' / CONFIG_NAME
resume_path = CHECKPOINT_DIR / 'latest.pt'

import importlib
importlib.invalidate_caches()
for module_name in [
    'ml.residual_unet.config',
    'ml.residual_unet.dataset',
    'ml.residual_unet.train',
]:
    if module_name in sys.modules:
        importlib.reload(sys.modules[module_name])

from ml.residual_unet.config import apply_overrides, load_config
import ml.residual_unet.train as train_module

config = apply_overrides(load_config(config_path), {
    'data.processed_dir': str(DATA_DIR),
    'data.batch_size': TRAIN_BATCH_SIZE,
    'data.num_workers': NUM_WORKERS,
    'data.prefetch_factor': PREFETCH_FACTOR,
    'data.pin_memory': True,
    'training.checkpoint_dir': str(CHECKPOINT_DIR),
    'training.log_csv': str(LOG_CSV),
    'training.progress_every': PROGRESS_EVERY,
})
resume = resume_path if resume_path.exists() else None

print('direct train() call')
print('config:', config_path)
print('checkpoint_dir:', CHECKPOINT_DIR)
print('log_csv:', LOG_CSV)
print('resume:', resume)
train_module.train(config, resume=resume)


## Evaluate Held-Out Terrain

Run separate HRRR-only and controlled-only reports so the controlled matrix does not hide weather-case weakness.


In [ ]:
heldout_sources = {
    'loveland': [
        'loveland_abasin_9p6_hrrr_monthly_202505_202604_v1',
        'loveland_abasin_9p6_controlled_9p6_15deg',
    ],
    'keystone': [
        'keystone_9p6_hrrr_monthly_202505_202604_v1',
        'keystone_9p6_controlled_9p6_15deg',
    ],
    'breck': [
        'breck_tenmile_9p6_hrrr_monthly_202505_202604_v1',
        'breck_tenmile_9p6_controlled_9p6_15deg',
    ],
}

heldout_key = 'loveland'
if 'keystone' in RUN_NAME:
    heldout_key = 'keystone'
elif 'breck' in RUN_NAME:
    heldout_key = 'breck'

import importlib
importlib.invalidate_caches()
for module_name in [
    'ml.residual_unet.dataset',
    'ml.residual_unet.evaluate',
]:
    if module_name in sys.modules:
        importlib.reload(sys.modules[module_name])

import ml.residual_unet.evaluate as evaluate_module

checkpoint = CHECKPOINT_DIR / 'best.pt'
assert checkpoint.exists(), checkpoint
for source in heldout_sources[heldout_key]:
    out = EVAL_ROOT / source
    print('evaluating', source)
    metrics = evaluate_module.evaluate(
        checkpoint,
        DATA_DIR,
        out,
        split='test',
        source_datasets=[source],
        batch_size=EVAL_BATCH_SIZE,
        num_workers=NUM_WORKERS,
        prefetch_factor=PREFETCH_FACTOR,
        max_figures=5,
    )
    print(source, json.dumps(metrics, indent=2))


## Package Results Back To GCS

The checkpoints, training log, metrics, sample CSVs, and figures are written under `MyDrive/windninja_ml/results/<run_name>/`. This final cell syncs that result directory back to the project bucket for local inspection.


In [ ]:
print('Result files:')
for path in sorted(RESULT_DIR.rglob('*')):
    if path.is_file():
        print(path.relative_to(RESULT_DIR))

if SYNC_RESULTS_TO_GCS and IN_COLAB:
    subprocess.run(['gcloud', 'config', 'set', 'project', GCP_PROJECT], check=True)
    gcs_dir = f'gs://{GCS_BUCKET}/colab_results/{RUN_NAME}'
    subprocess.run(['gcloud', 'storage', 'rsync', '-r', str(RESULT_DIR), gcs_dir], check=True)
    subprocess.run(['gcloud', 'storage', 'ls', gcs_dir + '/'], check=True)
else:
    print('Skipping GCS sync')
